In [1]:
import pandas as pd
import numpy as np
import json
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import rdChemReactions
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem.Draw import IPythonConsole
from IPython.display import SVG, display
from rdkit.Chem.Fingerprints import FingerprintMols
from rdkit import DataStructs
from rdkit.Chem import MACCSkeys
from rdkit.Chem import AllChem
import pandas as pd
import rdchiral
import itertools
import pickle
import sys
from tqdm import tqdm 
from rdchiral.main import rdchiralRun, rdchiralRunText, rdchiralReaction, rdchiralReactants
import cobra
from itertools import chain, combinations
import sys
import multiprocessing
from functools import partial
import matplotlib.pyplot as plt
import multiprocessing
from tqdm import tqdm
sys.path.append('../')

from common import *

### input and output

In [2]:
# step1
#input
mnxreac_smile_atom_mapping_rules_file_path = '../../Data/retrosynthesis/MNXreaction_smiles_atommap_rules_filter_0.2.csv'
mnxreac_smile_atom_mapping_rules_file_path = '/home/wuke/project/microbiome–drug/Results/MNXreaction_smiles_atommap_rules_filter_0.2.csv'
# target_drug_path = '../../Data/Drug/drug_Javdan_2020.csv'
target_drug_path = '../../Data/Drug/drug_MDM-_Javdan_2020.csv'

###output

In [3]:
def process_rule(rule_smiles, mets_total_smiles):
    score = []
    smiles = []
    if 'C' in rule_smiles or 'c' in rule_smiles:
        for j in mets_total_smiles:
            current_score = calculate_similarity(rule_smiles, j)
            if current_score > 0.3:               ########################################################################
                score.append(current_score)
                smiles.append(j)
    else:
        smiles.append(rule_smiles)
        score.append(1)

    if len(score) == 0:
        return rule_smiles, [], []
    elif len(score) > 50:
        top40_data = sorted(zip(score, smiles), reverse=True)[:50]
    else:
        top40_data = sorted(zip(score, smiles), reverse=True)
    
    top40_scores, top40_smiles = zip(*top40_data)
    
    return rule_smiles, list(top40_smiles), list(top40_scores)

####################################################################################

def count_carbons(smiles):
    """计算一个 SMILES 字符串中碳 (C) 的个数"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return 0
    return sum(1 for atom in mol.GetAtoms() if atom.GetSymbol() == 'C')

# def process_rule(rule_smiles, mets_total_smiles):
#     score = []
#     smiles = []
    
#     # 计算 rule_smiles 的碳原子数
#     rule_c_count = count_carbons(rule_smiles)

#     if 'C' in rule_smiles or 'c' in rule_smiles:
#         for j in mets_total_smiles:
#             # 计算当前代谢物的碳原子数
#             j_c_count = count_carbons(j)
            
#             # 根据 mets_total_smiles 的碳原子数目，确定允许的碳数目差
#             if j_c_count < 10:
#                 allowed_diff = 1
#             elif j_c_count < 15:
#                 allowed_diff = 2
#             else:
#                 allowed_diff = 3
            
#             # 仅在碳数目差符合要求时计算相似度
#             if abs(rule_c_count - j_c_count) <= allowed_diff:
#                 current_score = calculate_similarity(rule_smiles, j)
#                 if current_score > 0.3:
#                     score.append(current_score)
#                     smiles.append(j)
    
#     else:
#         smiles.append(rule_smiles)
#         score.append(1)

#     if len(score) == 0:
#         return rule_smiles, [], []
#     elif len(score) > 20:
#         top20_data = sorted(zip(score, smiles), reverse=True)[:20]
#     else:
#         top20_data = sorted(zip(score, smiles), reverse=True)
    
#     top20_scores, top20_smiles = zip(*top20_data)
    
#     return rule_smiles, list(top20_smiles), list(top20_scores)

# def process_rule(rule_smiles, mets_total_smiles):
#     score = []
#     smiles = []
    
#     # 计算 rule_smiles 的碳原子数
#     rule_c_count = count_carbons(rule_smiles)

#     if 'C' in rule_smiles or 'c' in rule_smiles:
#         for j in mets_total_smiles:
#             # 计算当前代谢物的碳原子数
#             j_c_count = count_carbons(j)
            
#             # 计算相似度
#             current_score = calculate_similarity(rule_smiles, j)
            
#             # 计算罚分
#             if rule_c_count!=0:
#                 if rule_c_count > j_c_count:
#                     penalty = (rule_c_count - j_c_count) / (rule_c_count) /10.216
#                 elif rule_c_count < j_c_count:
#                     penalty = (j_c_count - rule_c_count) / (rule_c_count) /8.828
#                 else:
#                     penalty = 0
#             else:
#                 penalty = 0
            
#             # 应用罚分
#             adjusted_score = max(0, current_score - penalty)

#             # 仅保留相似度大于0.3的结果
#             if adjusted_score > -5:
#                 score.append(adjusted_score)
#                 smiles.append(j)
    
#     else:
#         smiles.append(rule_smiles)
#         score.append(1)

#     if len(score) == 0:
#         return rule_smiles, [], []
#     # elif len(score) > 100:
#     #     top20_data = sorted(zip(score, smiles), reverse=True)[:100]
#     else:
#         top20_data = sorted(zip(score, smiles), reverse=True)
    
#     top20_scores, top20_smiles = zip(*top20_data)
    
#     return rule_smiles, list(top20_smiles), list(top20_scores)

# def process_rule(rule_smiles, mets_total_smiles):
#     score = []
#     smiles = []
    
#     # 计算 rule_smiles 的碳原子数
#     rule_c_count = count_carbons(rule_smiles)

#     if 'C' in rule_smiles or 'c' in rule_smiles:
#         for j in mets_total_smiles:
#             # 计算当前代谢物的碳原子数
#             j_c_count = count_carbons(j)
            
#             # 计算相似度
#             current_score = calculate_similarity(rule_smiles, j)
# # 最小满足条件的 N: 21.8108108108, M: 2.4864864865
#             # 计算罚分
#             if rule_c_count!=0:
#                 if rule_c_count > j_c_count:
#                     penalty = (rule_c_count - j_c_count) / (rule_c_count) /2.4864864865
#                 elif rule_c_count < j_c_count:
#                     penalty = (j_c_count - rule_c_count) / (rule_c_count) /21.8108108108
#                 else:
#                     penalty = 0
#             else:
#                 penalty = 0
            
#             # 应用罚分
#             adjusted_score = max(0, current_score - penalty)

#             # 仅保留相似度大于0.3的结果
#             if adjusted_score > 0.1:
#                 score.append(adjusted_score)
#                 smiles.append(j)
    
#     else:
#         smiles.append(rule_smiles)
#         score.append(1)

#     if len(score) == 0:
#         return rule_smiles, [], []
#     # elif len(score) > 20:
#         # top20_data = sorted(zip(score, smiles), reverse=True)[:20]
#     else:
#         top20_data = sorted(zip(score, smiles), reverse=True)
    
#     top20_scores, top20_smiles = zip(*top20_data)
    
#     return rule_smiles, list(top20_smiles), list(top20_scores)

In [4]:
# for index,row in retrosys_smiles_calculate_similarity_pd.iterrows():
#     if row['rules_smiles'] not in row['smiles_in_mets_total_smiles']:
#         retrosys_smiles_calculate_similarity_pd.loc[index,'smiles_in_mets_total_smiles'] = row['smiles_in_mets_total_smiles'].append(row['rules_smiles'])

In [5]:
def process_retrorule(index_row, mets_total_smiles,retrosys_smiles_calculate_similarity_pd,heterologous_met_smiles):
    rxndb_list_tmp = []
    # newdbSmiles_tmp = []
    failedrxn_tmp = []
    index, row = index_row
    rxn_id, ECnumber, reactants, products = row['MNX_ID'], row['classifs'], row['substrate_smiles'], row['product_smiles']
    deprecated_equ_smiles, reactantSMARTs, productSMARTs, rule = row['deprecated_equ_smiles'], row['ReactantsSMARTs'], row['ProductSMARTs'], row['RetroRules']
    deprecated_equ_smiles = deprecated_equ_smiles.split('>>')
    if rxn_id not in ['MNXR101882', 'MNXR101884', 'MNXR101879', 'MNXR101885', 'MNXR101886', 'MNXR101887',
                        'MNXR103528', 'MNXR103575']:  # ATP NADH conversion stuff
        productSMARTs = productSMARTs.split('.')
        products = products.split('.')
        new_products_smiles = []
        scores = []
        
        if len(productSMARTs) == len(products):
            for x in range(len(productSMARTs)):  # prepare mets with substructure and sort the reactants combination
                new_products_smile_one,score_one = get_calculate_similarity(products[x],retrosys_smiles_calculate_similarity_pd)
                new_products_smiles.append(new_products_smile_one)
                scores.append(score_one)

            product_prepared = []
            scores_prepared = []
            for lst, score_lst in zip(new_products_smiles, scores):
                if heterologous_met_smiles in lst:
                    if len(new_products_smiles) == 1:  # if only one sub list
                        product_prepared.append([heterologous_met_smiles])
                        scores_prepared.append(score_lst)
                    else:
                        other_lists = [sub_lst for sub_lst in new_products_smiles if sub_lst != lst]
                        tmp_product = list(itertools.product([heterologous_met_smiles], *other_lists))
                        product_prepared += tmp_product

                        tmp_scores = list(itertools.product([score_lst[new_products_smiles.index(lst)]], *scores))
                        scores_prepared += tmp_scores
            # print('product_prepared',product_prepared,len(product_prepared))
            for m in range(0, len(product_prepared)):  # apply the rule and output the rxn
                smile = product_prepared[m]
                score = scores_prepared[m]
                
                if len(smile) > 1:
                    product_smile = '.'.join(smile)
                else:
                    product_smile = smile[0]

                retrorule_tmp = rule.split('>>')
                retrorule_tmp = '(' + retrorule_tmp[0] + ')>>' + '(' + retrorule_tmp[1] + ')'            
                product_smile_tmp = product_smile
                # print('retrorule_tmp',retrorule_tmp)
                # print('product_smile',product_smile)
                try:
                    # print('product_smile_tmp',product_smile_tmp)
                    product_mol = Chem.MolFromSmiles(product_smile_tmp)
                    reaction = AllChem.ReactionFromSmarts(retrorule_tmp)
                    reactant_smiles = reaction.RunReactants((product_mol,))
                    smiles = [Chem.MolToSmiles(x[0]) for x in reactant_smiles]
                    reactant_smiles = list(set(smiles))

                    for reactant_smile in reactant_smiles:
                        if len(deprecated_equ_smiles) > 0:
                            if deprecated_equ_smiles[0] == '':
                                reactant_smile_final = reactant_smile
                            else:
                                reactant_smile_final = reactant_smile + '.' + deprecated_equ_smiles[0]
                            if deprecated_equ_smiles[1] == '':
                                product_smile_final = product_smile_tmp
                            else:
                                product_smile_final = product_smile_tmp + '.' + deprecated_equ_smiles[1]
                        rxn_smiles = reactant_smile + '>>' + product_smile_tmp
                        rxn_smiles_final = reactant_smile_final + '>>' + product_smile_final
                        
                        tmp = {
                            'EC number': ECnumber,
                            'rule': rule,
                            'templateID': rxn_id,
                            'templateSubstrate': products,
                            'rxn_smiles_basic': rxn_smiles,
                            'rxn_smiles_final': rxn_smiles_final,
                            'reactant_smile': reactant_smile,
                            'productsmile': product_smile_tmp,
                            'similarity': score
                        }
                        if check_rule(tmp['rxn_smiles_basic'])=='blance':
                            rxndb_list_tmp.append(tmp)  
                            # mets_tmp = reactant_smile.split('.')      
                            # for reactant in mets_tmp:             
                            #     if reactant not in mets_total_smiles and reactant not in newdbSmiles_tmp:         
                            #         newdbSmiles_tmp.append(reactant)        
                except:
                    failedrxn_tmp.append(rxn_id)
                
    return(rxndb_list_tmp,failedrxn_tmp)

In [6]:
def process_ec_list(ec_list,topn):
    processed_set = set()
    
    for ec in ec_list:
        for sub_ec in ec.split(';'):
            parts = sub_ec.strip().split('.')
            if len(parts) >= 3:  # Only process EC numbers with at least 3 parts
                processed_set.add('.'.join(parts[:topn]))
    
    return list(processed_set)

# Example usage
input_ec_numbers = ['4.1.3.-', '3.1.3.1', '1.1.3.100;2.1.4.2']
output_ec_numbers = process_ec_list(input_ec_numbers,3)
print(output_ec_numbers)

['2.1.4', '4.1.3', '1.1.3', '3.1.3']


In [7]:
def check_productsmile(smiles):#only keep smiles with carbon
    smiles_lst = smiles.split('.')
    smiles_lst = [x for x in smiles_lst if smiles_has_carbon(x)]
    return len(smiles_lst)

In [8]:
rules_product_smiles_lst = []
retrorules = pd.read_csv(mnxreac_smile_atom_mapping_rules_file_path)
# retrorules = retrorules[retrorules['MNX_ID'].str.contains('MNXR97814')]
for index,row in retrorules.iterrows():
    tmp  = row['product_smiles'].split('.')
    for i in tmp:
        if i not in rules_product_smiles_lst:
            rules_product_smiles_lst.append(i)
print('rules_product_smiles_lst',len(rules_product_smiles_lst))

rules_product_smiles_lst 11047


In [9]:
target_drug = pd.read_csv(target_drug_path)
# target_drug.rename(columns={'smiles':'drug_smiles'},inplace=True)
target_drug['drug_smiles'] = target_drug['drug_smiles'].apply(lambda x:x.replace('.Cl','').replace('.[Na+]',''))
target_drug = target_drug[target_drug['drug_smiles'].apply(lambda x:normalize_smiles(x)!=None)]
# target_drug = target_drug[target_drug['drug_name']=='digitoxin']
# target_drug = target_drug.loc[0:2,:]
# target_drug = target_drug.iloc[error_index[300:],:].reset_index(drop=True)
target_drug

,drug_name,drug_smiles
0,Clindamycin·HCl,CCC[C@@H]1C[C@H](N(C1)C)C(=O)N[C@@H]([C@@H]2[C...
1,Felbamate,C1=CC=C(C=C1)C(COC(=O)N)COC(=O)N
2,Cyclosporine A,CC[C@H]1C(=O)N(CC(=O)N([C@H](C(=O)N[C@H](C(=O)...
3,Donepezil·HCl,COC1=C(C=C2C(=C1)CC(C2=O)CC3CCN(CC3)CC4=CC=CC=...
4,Lincomycin·HCl,CCC[C@@H]1C[C@H](N(C1)C)C(=O)N[C@@H]([C@@H]2[C...
...,...,...
376,Pemirolast Potassium,CC1=CC=CN2C1=NC=C(C2=O)C3=NN=N[N-]3.[K+]
377,Perindopril Erbumine,CCC[C@@H](C(=O)OCC)N[C@@H](C)C(=O)N1[C@H]2CCCC...
378,Perphenazine,C1CN(CCN1CCCN2C3=CC=CC=C3SC4=C2C=C(C=C4)Cl)CCO
379,Phenylephrine,CNC[C@@H](C1=CC(=CC=C1)O)O


In [10]:
# for index,row in target_drug.iterrows():
#     print('====================',row['drug_name'],'====================')
#     draw_smiles(row['drug_smiles'])
#     draw_smiles(row['product_smiles'])
# break

In [11]:
# 读取gut clean ec预测结果
with open('../../Results/AGORA2_strain_eclist_top3.pkl', 'rb') as f:
    loaded_top3 = pickle.load(f)

with open('../../Results/AGORA2_strain_eclist_top4.pkl', 'rb') as f:
    loaded_top4 = pickle.load(f)

print("读取完成！")
print(f"Top3 列表长度：{len(loaded_top3)}")
print(f"Top4 列表长度：{len(loaded_top4)}")

读取完成！
Top3 列表长度：252
Top4 列表长度：4948


## byproduct prediction

In [ ]:
result_num = {'drug_name':[],
              'reaction_num':[],
              'ec4_num':[],
              'ec4_list':[],
              'ec3_num':[],
              'ec3_list':[],
            #   'success_predict':[],
            #   'success_ecnumber':[],
              }
# filter_info = {'drug_carbon':[],
#                'success_carbon':[],
#                'error_carbon':[],
#                'success_sim_score':[],
#                'error_sim_score':[],
#                }

for index,row in tqdm(target_drug.iterrows(),total=len(target_drug)):
    print(row['drug_name'])
    heterologous_met_smiles = row['drug_smiles']
    # heterologous_met_smiles = 'FC1=CNC(=O)NC1=O'
    # heterologous_met_smiles = 'N[C@@H](CC1=CC(O)=C(O)C=C1)C(O)=O'
    # heterologous_met_smiles = '[NH3+][C@@H](Cc1ccc(O)c(O)c1)C(=O)[O-]' ######## L-dopa
    heterologous_met_smiles = normalize_smiles(heterologous_met_smiles)
    # model_met = target_drug['smiles'].to_list()
    # model_met = [heterologous_met_smiles,heterologous_met_smiles,heterologous_met_smiles,'O=C([O-])CCC(=O)C(=O)[O-]']
    model_met = [heterologous_met_smiles,heterologous_met_smiles,heterologous_met_smiles,'O=C([O-])CCC(=O)C(=O)[O-]',
                 'C[S+](CC[C@@H](C(=O)[O-])N)C[C@@H]1[C@H]([C@H]([C@@H](O1)N2C=NC3=C(N=CN=C32)N)O)O',
                 'C1=NC(=C2C(=N1)N(C=N2)[C@H]3[C@@H]([C@@H]([C@H](O3)CSCC[C@@H](C(=O)O)N)O)O)N',
                 'O=C(O)[C@H]1O[C@@H](O)[C@H](O)[C@@H](O)[C@H]1O',# β-D-葡萄糖醛酸
                 'CC(=O)SCCNC(=O)CCNC(=O)[C@@H](C(C)(C)COP(=O)(O)OP(=O)(O)OC[C@@H]1[C@H]([C@H]([C@@H](O1)N2C=NC3=C(N=CN=C32)N)O)OP(=O)(O)O)O', # Acetyl-CoA
                 ]
    new_met = []

    # add heterologous_met_smiles to model_met list
    new_model_and_heterologous_met = model_met + new_met
    new_model_and_heterologous_met.append(heterologous_met_smiles)
    new_model_and_heterologous_met = [smile for smile in new_model_and_heterologous_met if '.' not in smile]
    new_model_and_heterologous_met = [smile for smile in new_model_and_heterologous_met if 'C' in smile or 'c' in smile] ###只筛第一轮
    print(len(new_model_and_heterologous_met))
    #calculate similarity
    num_processes = 100
    pool = multiprocessing.Pool(num_processes)
    retrosys_smiles_calculate_similarity = {'rules_smiles': [],
                                            'smiles_in_mets_total_smiles': [],
                                            'scores': []}


    process_rule_partial = partial(process_rule, mets_total_smiles=new_model_and_heterologous_met)
    for result in tqdm(pool.imap(process_rule_partial, rules_product_smiles_lst, chunksize=100), total=len(rules_product_smiles_lst)):
        if result[0] not in retrosys_smiles_calculate_similarity['rules_smiles']:
            retrosys_smiles_calculate_similarity['rules_smiles'].append(result[0]) 
            retrosys_smiles_calculate_similarity['smiles_in_mets_total_smiles'].append(result[1]) 
            retrosys_smiles_calculate_similarity['scores'].append(result[2]) 
    pool.close()
    pool.join()

    retrosys_smiles_calculate_similarity_pd = pd.DataFrame(retrosys_smiles_calculate_similarity)
    
##############################################
    # for index4, row4 in retrosys_smiles_calculate_similarity_pd.iterrows():
    #     if row4['rules_smiles'] not in row4['smiles_in_mets_total_smiles']:
    #         # 修改 DataFrame 中嵌套的列表
    #         retrosys_smiles_calculate_similarity_pd.at[index4, 'smiles_in_mets_total_smiles'].append(row4['rules_smiles'])
##############################################
    rxndb_list = [] 
    # newdbSmiles = []
    failedrxn = []
    num_processes =100  #  adjust the number of processes according to your needs
    pool = multiprocessing.Pool(num_processes)
    # Execute in parallel using process pool

    # Create a partial function that passes mets_total_smiles and mets_total_mol as default parameters to the process_retrorule function
    process_retrorule_partial = partial(process_retrorule, mets_total_smiles=new_model_and_heterologous_met,retrosys_smiles_calculate_similarity_pd = retrosys_smiles_calculate_similarity_pd,heterologous_met_smiles = heterologous_met_smiles)

    # Using the process pool to execute in parallel, the chunksize parameter is placed inside the imap function.
    for result in tqdm(pool.imap(process_retrorule_partial, retrorules.iterrows(), chunksize=100), total=len(retrorules)):
        # if result == ([], [], []):
        #     pass
        # else:
        #     print(result)
        if result[0] :  #  check if the result is not empty
            rxndb_list.extend(result[0]) 
        # if result[1] is not None:
        #     newdbSmiles.extend(result[1])
        if result[1]:
            failedrxn.extend(result[1])  

    pool.close()
    pool.join()

    rxndb = {}
    for i, rxn_dict in enumerate(rxndb_list):
        # generate a unique key for each reaction, for example "rxn0"，"rxn1"，"rxn2"，...
        key = f'rxn{i+1}'
        # merge to the dictionary
        rxndb[key] = rxn_dict
    # with open(rxndb_path, 'w') as json_file:
    #     json.dump(rxndb, json_file)

    # with open(rxndb_path, 'r') as f:
    #     rxndb = json.load(f)
    print('rxndb num',len(rxndb))
    print('failedrxn num',len(failedrxn))

    ecnumber_list = []
    for _,x in rxndb.items():
        ec_list = x['EC number'].split(';')
        ec_list = process_ec_list(ec_list,3)
        ecnumber_list += ec_list

    ecnumber_list = list(set(ecnumber_list))   
    result_df = {'ID':[],
            'reactant_smile':[],
            'productsmile':[],
            'rxn_smiles_basic':[],
            'templateID':[],
            'EC number':[],}

    for k,v in rxndb.items():
        result_df['ID'].append(k)
        result_df['reactant_smile'].append(v['reactant_smile'])
        result_df['productsmile'].append(v['productsmile'])
        result_df['rxn_smiles_basic'].append(v['rxn_smiles_basic'])
        result_df['templateID'].append(v['templateID'])
        result_df['EC number'].append(v['EC number'])

    result_df = pd.DataFrame({'ID': result_df['ID'],
                        'reactant_smile': result_df['reactant_smile'],
                        'productsmile': result_df['productsmile'],
                        'rxn_smiles_basic': result_df['rxn_smiles_basic'],
                        'templateID': result_df['templateID'],
                        'EC number': result_df['EC number'],
                        })

    result_df.drop_duplicates(subset=['reactant_smile', 'productsmile'], inplace=True)
    result_df.reset_index(drop=True, inplace=True)

    result_df = result_df[result_df['productsmile']!=heterologous_met_smiles+'.OO']
    result_df = result_df[result_df['productsmile']!='OO.'+ heterologous_met_smiles]
    result_df = result_df[result_df['productsmile']!=heterologous_met_smiles+'.O=C=O']
    result_df = result_df[result_df['productsmile']!=heterologous_met_smiles+'.O=S([O-])[O-]']
    result_df = result_df[result_df['productsmile']!=heterologous_met_smiles+'.[SH-]']
    result_df = result_df[result_df['productsmile']!=heterologous_met_smiles+'.O=P([O-])([O-])OP(=O)([O-])O']
    result_df = result_df[result_df['productsmile']!=heterologous_met_smiles+'.II']
    result_df = result_df[result_df['productsmile']!=heterologous_met_smiles+'.[NH4+]']
    result_df = result_df[result_df['productsmile']!=heterologous_met_smiles+'.[Br-]']

    result_df['productsmile_num'] = result_df['productsmile'].apply(lambda x: check_productsmile(x))
    result_df = result_df[result_df['productsmile_num']==1]


    # error_ec = ['1.2.1', '1.14.13', '1.14.14', '1.1.1', '3.1.3' ,'1.21.1','1.22.1','1.1.1','3.1.6'] #1.10.3#3.1.6
    error_ec = []
    # 过滤 DataFrame，使得 "EC number" 列中的每个 EC 号都属于 error_ec
    if len(result_df)>0:
        result_df['EC number'] = result_df['EC number'].apply(lambda x:x.split(';'))
        result_df = result_df[~result_df['EC number'].apply(lambda ec_list: all('.'.join(x.split('.')[0:3]) in error_ec for x in ec_list))]
        if len(result_df)>0:
            result_df['EC number'] = result_df['EC number'].apply(lambda x:';'.join(x))
            # 过滤 DataFrame，使得 "EC number" 列中的有 EC 号都属于 gut_ec
            # gut_ec_list_top3 = loaded_top3
            # result_df['EC number'] = result_df['EC number'].apply(lambda x:x.split(';'))
            # result_df = result_df[result_df['EC number'].apply(lambda ec_list: any('.'.join(x.split('.')[0:3]) in gut_ec_list_top3 for x in ec_list))]
            # result_df['EC number'] = result_df['EC number'].apply(lambda x:';'.join(x))        
    try:
        res = result_df['reactant_smile'].to_list()
        ec_lst = []
        for k,v in rxndb.items():
            if v['reactant_smile'] in res:
                ec_lst.append(v['EC number'])
    except:
        ec_lst = []
        pass
    # print(row['name'],len(result_df))
    # print(row['name'],ec_lst)
    print('result_df',len(result_df))
    print(ec_lst)
    ec_lst_4 = process_ec_list(ec_lst,4)
    ec_lst_3 = process_ec_list(ec_lst,3)
    print(len(ec_lst_3))
    print(len(ec_lst_4))
    
    result_num['drug_name'].append(row['drug_name'])
    result_num['reaction_num'].append(len(result_df))
    result_num['ec4_num'].append(len(ec_lst_4))
    result_num['ec4_list'].append(str(ec_lst_4))
    result_num['ec3_num'].append(len(ec_lst_3))
    result_num['ec3_list'].append(str(ec_lst_3))
    
    # success_predict = 0
    # success_ecnumber = []
    # success_ID = []
    # if row['product_smiles'] != 'unknown' and row['drug_name'] != 'Lactulose':
    #     target_smiles = row['product_smiles'].split('.')[0]
    #     for index2,row2 in result_df.iterrows():
    #         smiles_list = row2['reactant_smile'].split('.')
    #         if any(compare_smiles_inchikey(smiles,target_smiles) for smiles in smiles_list):
    #             success_predict =1
    #             success_ID.append(row2['ID'])
    #             tmp_ec_lst = row2['EC number'].split(';')
    #             success_ecnumber+=process_ec_list(tmp_ec_lst,3)
    
    # result_num['success_predict'].append(success_predict)
    # result_num['success_ecnumber'].append(str(list(set(success_ecnumber))))
    
    
    # drug_carbon = calculate_carbon_count(heterologous_met_smiles)
    # success_sim_score = []
    # success_carbon = []
    # error_carbon = []
    # error_sim_score = []
    # # for tmpid,tmp_result in rxndb.items():
    # #     if tmpid in result_df['ID']:
    # for index3,row3 in result_df.iterrows():
    #     if row3['ID'] in success_ID:
    #         success_sim_score.append(min(rxndb[row3['ID']]['similarity']))
    #         success_carbon.append(max(calculate_carbon_count(tmp_smiles) for tmp_smiles in rxndb[row3['ID']]['templateSubstrate']))
    #     else:
    #         error_sim_score.append(min(rxndb[row3['ID']]['similarity']))
    #         error_carbon.append(max(calculate_carbon_count(tmp_smiles) for tmp_smiles in rxndb[row3['ID']]['templateSubstrate']))

    # filter_info['drug_carbon'].append(drug_carbon)
    # filter_info['success_carbon'].append(success_carbon)
    # filter_info['error_carbon'].append(error_carbon)
    # filter_info['success_sim_score'].append(success_sim_score)
    # filter_info['error_sim_score'].append(error_sim_score)
    
    # # 保存结果
    # # 保存 drug_degration_ec 到 JSON 文件
    # drug_degration_ec_path = os.path.join(output_dir, 'drug_degration_ec.json')
    # with open(drug_degration_ec_path, 'w') as json_file:
    #     json.dump(drug_degration_ec, json_file, indent=4)

    # # 保存 result_df 到 CSV 文件
    # result_df_path = os.path.join(output_dir, 'result_df.csv')
    # # 保存结果时，使用追加模式（mode='a'）
    # result_df.to_csv(result_df_path, index=False, mode='a', header=not os.path.exists(result_df_path))

  0%|          | 0/381 [00:00<?, ?it/s]

Clindamycin·HCl
9


100%|██████████| 11047/11047 [00:07<00:00, 1387.83it/s]


In [44]:
result_df

,ID,reactant_smile,productsmile,rxn_smiles_basic,templateID,EC number,productsmile_num


In [45]:
result_df

,ID,reactant_smile,productsmile,rxn_smiles_basic,templateID,EC number,productsmile_num


In [46]:
rxndb

{}

In [47]:
result_df

,ID,reactant_smile,productsmile,rxn_smiles_basic,templateID,EC number,productsmile_num


In [48]:
result_num_df = pd.DataFrame(result_num)
result_num_df['ec3_list'] = result_num_df['ec3_list'].apply(lambda x:ast.literal_eval(x))
result_num_df['ec3_list'] = result_num_df['ec3_list'].apply(lambda x:[ec for ec in x if '-' not in ec])
result_num_df['ec3_num'] = result_num_df['ec3_list'].apply(lambda x:len(x))

result_num_df['ec4_list'] = result_num_df['ec4_list'].apply(lambda x:ast.literal_eval(x))
result_num_df['ec4_list'] = result_num_df['ec4_list'].apply(lambda x:[ec for ec in x if '-' not in ec])
result_num_df['ec4_num'] = result_num_df['ec4_list'].apply(lambda x:len(x))

result_num_df

,drug_name,reaction_num,ec4_num,ec4_list,ec3_num,ec3_list
0,Clindamycin·HCl,74,259,"[1.1.1.159, 1.1.1.357, 1.1.1.367, 3.1.5.1, 1.1...",36,"[3.4.11, 4.1.3, 3.3.2, 3.4.15, 6.2.1, 1.16.3, ..."
1,Felbamate,19,53,"[3.4.11.5, 1.11.2.6, 2.3.2.7, 3.4.15.6, 1.14.1...",25,"[3.4.11, 4.1.3, 3.10.1, 3.4.15, 6.2.1, 2.3.2, ..."
2,Cyclosporine A,134,315,"[1.1.1.357, 3.2.1.114, 3.1.3.68, 1.1.1.270, 6....",40,"[3.4.11, 4.1.3, 3.4.15, 1.16.3, 3.4.13, 1.14.2..."
3,Donepezil·HCl,73,84,"[1.11.2.6, 1.17.1.3, 1.14.14.1, 1.14.13.209, 1...",21,"[3.3.2, 6.2.1, 3.8.1, 1.14.16, 1.14.3, 1.14.15..."
4,Lincomycin·HCl,86,274,"[1.1.1.159, 1.1.1.357, 1.1.1.367, 3.1.5.1, 1.1...",36,"[3.4.11, 4.1.3, 3.3.2, 3.4.15, 6.2.1, 1.16.3, ..."
...,...,...,...,...,...,...
376,Pemirolast Potassium,0,0,[],0,[]
377,Perindopril Erbumine,0,0,[],0,[]
378,Perphenazine,43,60,"[1.1.1.23, 1.1.1.1, 3.1.5.1, 3.1.3.74, 1.14.14...",14,"[1.1.99, 1.1.1, 4.99.1, 1.1.3, 1.14.13, 3.1.3,..."
379,Phenylephrine,48,192,"[1.11.2.6, 1.17.1.3, 1.14.19.55, 3.1.5.1, 1.14...",31,"[2.8.2, 4.1.3, 1.16.3, 1.6.5, 1.14.16, 1.14.3,..."


In [49]:
result_num_df.to_csv('../../Results/ref2_mdm-_drug_deg_result_filter.csv',index=None)